In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import sys
sys.path.append(r'C:\Users\julia\OneDrive\Escritorio\Trabajo\building_ml_models_for_protein_science\src')

In [3]:
from building_models.commons_functions.parsers_commons import ParsersCommons
from building_models.utils.constants import COLUMNS_TO_WORK
from building_models.utils.utils_functions import UtilsFunctions
import pandas as pd

- Read doc and labels

In [4]:
path_export = "../../processed_dataset/"
path_input = "../../raw_dataset/"
metadata_file = "../../raw_dataset/raw_data_description.xlsx"
name_task = "antioxidant_classification"
name_source = "Butt et al"

In [5]:
df_independent = pd.read_excel(f"{path_input}/{name_source}/1-s2.0-S0022519319301602-mmc1.xlsx", sheet_name="Independent-Antioxidants-120", header=None)
df_independent = df_independent.rename(columns={0:'sequence'})
df_independent['label'] = 1

In [6]:
df_train = pd.read_excel(f"{path_input}/{name_source}/1-s2.0-S0022519319301602-mmc1.xlsx", sheet_name="Training-Antioxidants-250", header=None)
df_train = df_train.rename(columns={0:'sequence'})
df_train = df_train[~df_train['sequence'].str.contains('>')]
df_train['label'] = 1
df_train

,sequence,label
1,MTKGILLGDKFPDFRAETNEGFIPSFYDWIGKDSWAILFSHPRDFT...,1
3,MLPGLALLLLAAWTARALEVPTDGNAGLLAEPQIAMFCGRLNMHMN...,1
5,MAIALSSSSTITSITLQPKLKTIHGLGTVLPGYSVKSHFRSVSLRR...,1
7,MITSSKKIVSAMLSTSLWIGVASAAYAETTNVEAEGYSTIGGTYQD...,1
9,MANSGLWELITIGSAVRNVAKSYLKAEASSITAKQLYDASKITSSK...,1
...,...,...
491,SNAPLLLGKKAPNLYMTDTTGTYRYLYDVQAKYTILFFWDSQCGHC...,1
493,MSLAPGKAESDAPLVRTGALAPNFKLPTLSGENKSLAQYRGKIVLV...,1
495,MSLAVKPGEPLPDFLLLDPKGQPVTPATVSKPAVIVFWASWCTVCK...,1
497,MSLEENPAPDFTLNTLNGEVVKLSDLKGQVVIVNFWATWCPPCREE...,1


In [7]:
df_neg = pd.read_excel(f"{path_input}/{name_source}/1-s2.0-S0022519319301602-mmc1.xlsx", sheet_name="Sheet3", header=None)
df_neg = df_neg.rename(columns={1:'sequence'})
df_neg['label'] = 0
df_neg = df_neg[[ 'sequence', 'label']]
df_neg

,sequence,label
0,MERPWGAADGLSRWPHGLGLLLLLQLLPPSTLSQDRLDAPPPPAAP...,0
1,MKKIMLVFITLILVSLPIAQQTEAKDASAFNKENSISSMAPPASPP...,0
2,MHYCVLSAFLLLHLVTVALSLSTCSTLDMDQFMRKRIEAIRGQILS...,0
3,MTGAKRKKRSVLWGKMHTPHREDIKQWCKRRLPILEWAPQYNLKEN...,0
4,MLNVEPSFAEELRSSGVSLSATYLGSVPVVESINVMVSEMRVQVVS...,0
...,...,...
1570,MAEGEKNQDFTFKMESPSDSAVVLPSTPQASANPSSPYTNSSRKQP...,0
1571,MASALRPPRVPKPKGVLPSHYYESFLEKKGPCDRDYKKFWAGLQGL...,0
1572,MFPEQQKEEFVSVWVRDPRIQKEDFWHSYIDYEICIHTNSMCFTMK...,0
1573,MSNSAQSRRIRVTIVAADGLYKRDVFRFPDPFAVLTVDGEQTHTTT...,0


In [8]:
df_data = pd.concat([df_independent, df_train, df_neg], ignore_index=True)
df_data["label"] = df_data["label"].astype(int)
df_data

,sequence,label
0,MAAAAGRLLWSSVARHASAISRSISASTVLRPVASRRTCLTDILWS...,1
1,MAAAVGRLLRASVARHVSAIPWGISATAALRPAACGRTSLTNLLCS...,1
2,MAAKTGSQLERSISTIINVFHQYSRKYGHPDTLNKAEFKEMVNKDL...,1
3,MAALDAIREALPEPARDIKLNLQAVLQPGPLTPAQRWGVAVATAAA...,1
4,MAALKAGRGANWSLRAWRALGGIFWRKPPLLAPDLRALLTSGTPDS...,1
...,...,...
1940,MAEGEKNQDFTFKMESPSDSAVVLPSTPQASANPSSPYTNSSRKQP...,0
1941,MASALRPPRVPKPKGVLPSHYYESFLEKKGPCDRDYKKFWAGLQGL...,0
1942,MFPEQQKEEFVSVWVRDPRIQKEDFWHSYIDYEICIHTNSMCFTMK...,0
1943,MSNSAQSRRIRVTIVAADGLYKRDVFRFPDPFAVLTVDGEQTHTTT...,0


- Checking duplicates

In [9]:
df_consistent_duplicates, df_errors, df_unique = ParsersCommons.processing_duplicated(
    df_data, group_seq= "sequence",
    label_col= "label")
df_consistent_duplicates.shape, df_errors.shape, df_unique.shape
#no duplicates

((0, 0), (0, 0), (1945, 2))

- Checking labels

In [10]:
df_data["label"].value_counts()

label
0    1575
1     370
Name: count, dtype: int64

- Working with metadata


In [11]:
metadata_file = ParsersCommons.read_metadata(metadata_file, name_source=name_source, columns_to_select=COLUMNS_TO_WORK)
metadata_file.head()

,name dataset,name source,type source,static-dynamic,license,reports constant updates,year of publication,last update date,download date,file format,protein format,category dataset,task,obtaining negative dataset,obtaining positive dataset,repository or server,publication,unit of measurement
44,1-s2.0-S0022519319301602-mmc1.xlsx,Butt et al,Dataset,Static,No information,No,2019,2019-07-21,2025-09-07,xlsx,Sequence,Enzyme/protein classification,Antioxidant,"Sampling from UniProt, Manually curated databa...","Sampling from UniProt, Previously published mo...",Supplementary Material,https://www.sciencedirect.com/science/article/...,No information


In [12]:
dict_metadata = ParsersCommons.create_metadata_from_file(metadata_file)
dict_metadata

{'name dataset': '1-s2.0-S0022519319301602-mmc1.xlsx',
 'name source': 'Butt et al',
 'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'No information',
 'reports constant updates': 'No',
 'year of publication': 2019,
 'last update date': Timestamp('2019-07-21 00:00:00'),
 'download date': Timestamp('2025-09-07 00:00:00'),
 'file format': 'xlsx',
 'protein format': 'Sequence',
 'category dataset': 'Enzyme/protein classification',
 'task': 'Antioxidant',
 'obtaining negative dataset': 'Sampling from UniProt, Manually curated database, Sampling from Swiss-Prot',
 'obtaining positive dataset': 'Sampling from UniProt, Previously published model dataset, Sampling from Swiss-Prot',
 'repository or server': 'Supplementary Material',
 'publication': 'https://www.sciencedirect.com/science/article/pii/S0022519319301602?via%3Dihub',
 'unit of measurement': 'No information',
 'number_of_sources': 1,
 'processing_date': '2026-04-08 17:00:02'}

In [13]:
dict_metadata['number_of_records'] = df_data.shape[0]
dict_metadata['number_of_collected_sequences'] = df_data.shape[0]
dict_metadata['number_of_unique_sequences'] = df_data.shape[0]
dict_metadata['positive_examples'] = df_data[df_data["label"] == 1].shape[0]
dict_metadata['negative_examples'] = df_data[df_data["label"] == 0].shape[0]
dict_metadata['number_of_sequences_with_errors'] = df_errors.shape[0]
dict_metadata

{'name dataset': '1-s2.0-S0022519319301602-mmc1.xlsx',
 'name source': 'Butt et al',
 'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'No information',
 'reports constant updates': 'No',
 'year of publication': 2019,
 'last update date': Timestamp('2019-07-21 00:00:00'),
 'download date': Timestamp('2025-09-07 00:00:00'),
 'file format': 'xlsx',
 'protein format': 'Sequence',
 'category dataset': 'Enzyme/protein classification',
 'task': 'Antioxidant',
 'obtaining negative dataset': 'Sampling from UniProt, Manually curated database, Sampling from Swiss-Prot',
 'obtaining positive dataset': 'Sampling from UniProt, Previously published model dataset, Sampling from Swiss-Prot',
 'repository or server': 'Supplementary Material',
 'publication': 'https://www.sciencedirect.com/science/article/pii/S0022519319301602?via%3Dihub',
 'unit of measurement': 'No information',
 'number_of_sources': 1,
 'processing_date': '2026-04-08 17:00:02',
 'number_of_records': 1945,
 'number_o

- Export data

In [14]:
UtilsFunctions.make_directory(f"{path_export}{name_task}/{name_source}")

In [15]:
UtilsFunctions.export_json(f"{path_export}{name_task}/{name_source}/metadata_{name_source}.json", dict_metadata)

In [16]:
df_data.to_csv(f"{path_export}{name_task}/{name_source}/processed_data.csv", index=False)